In [0]:

# ============================================================
# CELDA 1. CONFIGURACIÓN DEL ENTORNO BRONZE
# ============================================================

# Este notebook se utilizará para cargar los datos originales
# de vuelos BTS 2024-2025 a la capa Bronze de Databricks.

# Importamos funciones de PySpark para el procesamiento.
from pyspark.sql import functions as F

# Definimos el catálogo y el esquema donde guardaremos
# los datos originales de vuelos.
CATALOG = "workspace"
SCHEMA = "bronze_flights"

# Obtenemos la sesión Spark disponible en Databricks.
spark_session = spark

# Comprobamos que PySpark está funcionando.
print("Versión de Spark:", spark_session.version)

# Revisamos que el esquema Bronze exista y sea accesible.
spark_session.sql(
    f"SHOW TABLES IN {CATALOG}.{SCHEMA}"
).show(truncate=False)

print("Conexión con Bronze verificada correctamente.")


In [0]:
# ============================================================
# CELDA 2. LIMPIEZA DE LA TABLA TEMPORAL DE PRUEBA
# ============================================================

# Eliminamos únicamente la tabla que utilizamos para
# verificar los permisos del tercer integrante.
# No modificamos los esquemas ni los datos del proyecto.

spark.sql("""
    DROP TABLE IF EXISTS
    workspace.bronze_flights.test_access_skraeder
""")

# Comprobamos que Bronze ya no tenga tablas de prueba.
spark.sql(
    "SHOW TABLES IN workspace.bronze_flights"
).show(truncate=False)

print("Limpieza de tablas de prueba completada.")

In [0]:

# ============================================================
# CELDA 3. PRUEBA DE DESCARGA BTS - ENERO 2024
# ============================================================

# Antes de automatizar los 24 meses, comprobaremos
# que Databricks pueda acceder a la fuente oficial BTS.
#
# En Bronze utilizaremos TODOS los vuelos de cada mes.
# No aplicaremos muestreo ni limpieza en esta etapa.

import requests
import zipfile
import io
import time

# Definimos el periodo de prueba.
ANIO = 2024
MES = 1

# Construimos la URL oficial del archivo mensual.
nombre_archivo = (
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    f"1987_present_{ANIO}_{MES}.zip"
)

url = f"https://transtats.bts.gov/PREZIP/{nombre_archivo}"

print("Fuente:", url)
print("Comprobando acceso desde Databricks...")

inicio = time.time()

try:
    # Descargamos el ZIP para verificar la conexión.
    # Todavía no creamos ninguna tabla Delta.
    respuesta = requests.get(
        url,
        timeout=120
    )

    respuesta.raise_for_status()

    # Verificamos que el contenido descargado sea un ZIP.
    archivo_zip = io.BytesIO(respuesta.content)

    with zipfile.ZipFile(archivo_zip) as comprimido:

        archivos_csv = [
            nombre
            for nombre in comprimido.namelist()
            if nombre.lower().endswith(".csv")
        ]

        if not archivos_csv:
            raise ValueError(
                "El ZIP no contiene archivos CSV."
            )

        print("Descarga exitosa.")
        print("Archivo CSV:", archivos_csv[0])
        print(
            "Tamaño del ZIP:",
            round(len(respuesta.content) / (1024**2), 2),
            "MB"
        )

        print(
            "Tiempo transcurrido:",
            round(time.time() - inicio, 2),
            "segundos"
        )

except requests.exceptions.RequestException as error:

    print("No se pudo descargar el archivo desde Databricks.")
    print("Detalle:", error)

except zipfile.BadZipFile:

    print("El archivo recibido no tiene un formato ZIP válido.")


In [0]:

# ============================================================
# CELDA 4. ALMACENAMIENTO Y LECTURA DEL ARCHIVO BTS
# ============================================================

# En esta etapa guardamos el CSV original de enero 2024
# en un Volume de Databricks para conservarlo.
#
# Después utilizaremos PySpark para leer todos los vuelos.
# No eliminamos registros ni realizamos muestreo.

import io
import zipfile
import shutil
from pathlib import Path

# Creamos un Volume administrado para almacenar
# los archivos originales descargados de BTS.
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.bronze_flights.bts_source_files
""")

# Definimos dónde guardaremos los archivos originales.
VOLUME_PATH = Path(
    "/Volumes/workspace/bronze_flights/bts_source_files"
)

# Creamos una carpeta para los archivos CSV.
CSV_PATH = VOLUME_PATH / "csv"
CSV_PATH.mkdir(parents=True, exist_ok=True)

# Utilizamos el ZIP descargado en la celda anterior.
# La variable 'respuesta' contiene el archivo completo.
with zipfile.ZipFile(io.BytesIO(respuesta.content)) as zip_bts:

    # Identificamos el archivo CSV dentro del ZIP.
    csvs = [
        nombre for nombre in zip_bts.namelist()
        if nombre.lower().endswith(".csv")
    ]

    if not csvs:
        raise ValueError("No se encontró un CSV en el ZIP.")

    # Definimos un nombre sencillo para enero de 2024.
    archivo_destino = CSV_PATH / "bts_2024_01.csv"

    # Extraemos el contenido completo.
    with zip_bts.open(csvs[0]) as origen:
        with open(archivo_destino, "wb") as destino:
            shutil.copyfileobj(origen, destino)

print("CSV original guardado en:", archivo_destino)

# Leemos el CSV completo usando PySpark.
# Conservamos inicialmente las columnas como texto,
# para no alterar los valores originales en Bronze.
df_bts = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "false")
    .csv(str(archivo_destino))
)

# Contamos los registros del archivo completo.
total_vuelos = df_bts.count()

print("\nLECTURA BTS - ENERO 2024")
print("Total de vuelos:", f"{total_vuelos:,}")
print("Número de columnas:", len(df_bts.columns))

# Mostramos cinco registros para comprobar la lectura.
display(df_bts.limit(5))


In [0]:

# ============================================================
# CELDA 5. CREACIÓN DE LA TABLA DELTA BRONZE - ENERO 2024
# ============================================================

from pyspark.sql import functions as F

# Definimos la ubicación del CSV que guardamos en la Celda 4.
# Lo recuperamos directamente desde el Volume de Databricks.
# De esta manera no dependemos de variables de otras celdas.

RUTA_CSV = (
    "/Volumes/workspace/bronze_flights/"
    "bts_source_files/csv/bts_2024_01.csv"
)

# Leemos todos los registros de enero de 2024 con PySpark.
# Conservamos las columnas como texto, ya que Bronze
# debe mantener los datos originales sin transformaciones.
df_bts = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(RUTA_CSV)
)

# Agregamos metadatos para identificar el origen
# y el momento en que se incorporaron los datos.
df_bronze = (
    df_bts
    .withColumn("_source_year", F.lit(2024))
    .withColumn("_source_month", F.lit(1))
    .withColumn("_source_file", F.lit("bts_2024_01.csv"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Definimos la tabla Delta que almacenará los vuelos.
TABLA_BRONZE = "workspace.bronze_flights.bts_flights_raw"

# Inicializamos la tabla con enero de 2024.
# IMPORTANTE: overwrite solo se utilizará en esta etapa.
# Para los siguientes meses haremos una carga incremental.
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLA_BRONZE)
)

# Comprobamos los registros almacenados en Delta Lake.
total_bronze = spark.table(TABLA_BRONZE).count()

print("Tabla Delta creada:", TABLA_BRONZE)
print("Registros almacenados:", f"{total_bronze:,}")
print("Número de columnas:", len(spark.table(TABLA_BRONZE).columns))


In [0]:

# ============================================================
# CELDA 6. CARGA INCREMENTAL BTS - FEBRERO 2024
# ============================================================

# Objetivo:
# Descargar febrero de 2024, guardar el archivo original
# y agregar todos sus vuelos a la tabla Delta existente.
#
# No usamos muestras ni sobrescribimos enero.

import requests
import zipfile
import shutil
import time

from pathlib import Path
from pyspark.sql import functions as F

ANIO = 2024
MES = 2

TABLA_BRONZE = "workspace.bronze_flights.bts_flights_raw"

VOLUME = Path(
    "/Volumes/workspace/bronze_flights/bts_source_files"
)

# Comprobamos si el mes ya está cargado.
# Esto evita agregar dos veces los mismos vuelos.
mes_existente = (
    spark.table(TABLA_BRONZE)
    .filter(
        (F.col("_source_year") == ANIO) &
        (F.col("_source_month") == MES)
    )
    .limit(1)
    .count()
)

if mes_existente > 0:
    print(f"El mes {ANIO}-{MES:02d} ya está cargado.")

else:

    inicio = time.time()

    # Construimos la dirección oficial del ZIP de BTS.
    nombre_zip = (
        "On_Time_Reporting_Carrier_On_Time_Performance_"
        f"1987_present_{ANIO}_{MES}.zip"
    )

    url = f"https://transtats.bts.gov/PREZIP/{nombre_zip}"

    carpeta_zip = VOLUME / "zip"
    carpeta_csv = VOLUME / "csv"

    carpeta_zip.mkdir(parents=True, exist_ok=True)
    carpeta_csv.mkdir(parents=True, exist_ok=True)

    ruta_zip = carpeta_zip / nombre_zip
    ruta_csv = carpeta_csv / f"bts_{ANIO}_{MES:02d}.csv"

    # Descargamos el archivo completo por bloques.
    if not ruta_zip.exists() or not zipfile.is_zipfile(ruta_zip):

        temporal = ruta_zip.with_suffix(".part")

        with requests.get(
            url, stream=True, timeout=(30, 180)
        ) as respuesta:

            respuesta.raise_for_status()

            with open(temporal, "wb") as archivo:
                for bloque in respuesta.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if bloque:
                        archivo.write(bloque)

        if not zipfile.is_zipfile(temporal):
            temporal.unlink(missing_ok=True)
            raise ValueError("Descarga ZIP inválida.")

        temporal.replace(ruta_zip)

    # Extraemos el CSV completo.
    with zipfile.ZipFile(ruta_zip) as comprimido:

        csvs = [
            nombre for nombre in comprimido.namelist()
            if nombre.lower().endswith(".csv")
        ]

        if not csvs:
            raise ValueError("El ZIP no contiene un CSV.")

        with comprimido.open(csvs[0]) as origen:
            with open(ruta_csv, "wb") as destino:
                shutil.copyfileobj(origen, destino)

    # Leemos todos los vuelos con PySpark.
    df_mes = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(str(ruta_csv))
    )

    # Agregamos los mismos metadatos que tiene enero.
    df_mes = (
        df_mes
        .withColumn("_source_year", F.lit(ANIO))
        .withColumn("_source_month", F.lit(MES))
        .withColumn(
            "_source_file",
            F.lit(ruta_csv.name)
        )
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
    )

    # Verificamos que las columnas coincidan con la tabla.
    columnas_tabla = spark.table(TABLA_BRONZE).columns

    if set(df_mes.columns) != set(columnas_tabla):
        raise ValueError(
            "Las columnas del nuevo mes no coinciden "
            "con el esquema Bronze. Revisar antes de agregar."
        )

    df_mes = df_mes.select(*columnas_tabla)

    # APPEND agrega registros sin borrar los anteriores.
    (
        df_mes.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_BRONZE)
    )

    print(f"Febrero {ANIO} cargado correctamente.")
    print(
        "Tiempo de carga:",
        round(time.time() - inicio, 2),
        "segundos"
    )

# Revisamos los registros almacenados por mes.
resumen = (
    spark.table(TABLA_BRONZE)
    .groupBy("_source_year", "_source_month")
    .count()
    .orderBy("_source_year", "_source_month")
)

display(resumen)

print(
    "Total acumulado:",
    f"{spark.table(TABLA_BRONZE).count():,}"
)


In [0]:

# ============================================================
# CELDA 7. AMPLIACIÓN DE BRONZE: BTS 2022 Y 2023
# ============================================================

# Objetivo:
# Descargar todos los vuelos de 2022 y 2023 y agregarlos
# a nuestra tabla Delta existente de Bronze.
#
# No utilizamos muestras.
# No eliminamos ni sobrescribimos los vuelos de 2024 y 2025.
#
# Si un mes ya está en la tabla, lo omitimos para evitar
# cargar nuevamente sus registros.

import requests
import zipfile
import shutil
import time

from pathlib import Path
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

TABLA_BRONZE = "workspace.bronze_flights.bts_flights_raw"

VOLUME = Path(
    "/Volumes/workspace/bronze_flights/bts_source_files"
)

CARPETA_ZIP = VOLUME / "zip"
CARPETA_CSV = VOLUME / "csv"

CARPETA_ZIP.mkdir(parents=True, exist_ok=True)
CARPETA_CSV.mkdir(parents=True, exist_ok=True)

# Solamente procesaremos 2022 y 2023.
# Los años 2024 y 2025 ya están almacenados en Bronze.
periodos = [
    (anio, mes)
    for anio in [2022, 2023]
    for mes in range(1, 13)
]

# Obtenemos los meses que ya están cargados en Delta.
# Esto permite volver a ejecutar el código sin duplicar
# meses que se hayan procesado anteriormente.
meses_cargados = {
    (fila["_source_year"], fila["_source_month"])
    for fila in (
        spark.table(TABLA_BRONZE)
        .select("_source_year", "_source_month")
        .distinct()
        .collect()
    )
}

# Recuperamos las columnas de la tabla original.
# Todos los meses deben conservar la misma estructura.
columnas_bronze = spark.table(TABLA_BRONZE).columns

resultados = []
errores = []

print("Iniciando ingesta histórica de BTS 2022-2023")
print("Meses por revisar:", len(periodos))
print("Meses previamente cargados:", len(meses_cargados))


# ------------------------------------------------------------
# 2. DESCARGA, LECTURA Y CARGA INCREMENTAL
# ------------------------------------------------------------

for anio, mes in periodos:

    periodo = f"{anio}-{mes:02d}"

    print(f"\nProcesando {periodo}...")

    # Revisamos si el mes ya existe antes de descargarlo.
    if (anio, mes) in meses_cargados:

        print("Mes ya cargado. Se omite.")

        resultados.append({
            "periodo": periodo,
            "estado": "Ya existente",
            "registros": None,
            "segundos": 0
        })

        continue

    inicio = time.time()

    try:

        # Construimos la URL oficial del archivo mensual BTS.
        nombre_zip = (
            "On_Time_Reporting_Carrier_On_Time_Performance_"
            f"1987_present_{anio}_{mes}.zip"
        )

        url = f"https://transtats.bts.gov/PREZIP/{nombre_zip}"

        ruta_zip = CARPETA_ZIP / nombre_zip
        ruta_csv = CARPETA_CSV / f"bts_{anio}_{mes:02d}.csv"

        # ----------------------------------------------------
        # 2.1. DESCARGAR EL ZIP ORIGINAL
        # ----------------------------------------------------

        # Si ya existe un ZIP válido, no lo descargamos otra vez.
        if not ruta_zip.exists() or not zipfile.is_zipfile(ruta_zip):

            temporal = ruta_zip.with_suffix(".part")

            for intento in range(1, 4):

                try:
                    temporal.unlink(missing_ok=True)

                    with requests.get(
                        url,
                        stream=True,
                        timeout=(30, 180)
                    ) as respuesta:

                        respuesta.raise_for_status()

                        with open(temporal, "wb") as archivo:

                            for bloque in respuesta.iter_content(
                                chunk_size=1024 * 1024
                            ):
                                if bloque:
                                    archivo.write(bloque)

                    if not zipfile.is_zipfile(temporal):
                        raise ValueError(
                            "El archivo descargado no es un ZIP válido."
                        )

                    temporal.replace(ruta_zip)
                    break

                except Exception:

                    temporal.unlink(missing_ok=True)

                    if intento == 3:
                        raise

                    print(
                        f"Reintentando descarga ({intento}/3)"
                    )

                    time.sleep(5)

        # ----------------------------------------------------
        # 2.2. EXTRAER EL CSV COMPLETO
        # ----------------------------------------------------

        # Extraemos todos los registros del CSV original.
        # Utilizamos un archivo temporal para evitar dejar
        # un CSV incompleto si se interrumpe la extracción.
        if not ruta_csv.exists():

            temporal_csv = ruta_csv.with_suffix(".part")

            temporal_csv.unlink(missing_ok=True)

            with zipfile.ZipFile(ruta_zip) as comprimido:

                csvs = [
                    archivo
                    for archivo in comprimido.namelist()
                    if archivo.lower().endswith(".csv")
                ]

                if not csvs:
                    raise ValueError(
                        "El ZIP no contiene archivos CSV."
                    )

                with comprimido.open(csvs[0]) as origen:

                    with open(temporal_csv, "wb") as destino:
                        shutil.copyfileobj(origen, destino)

            temporal_csv.replace(ruta_csv)

        # ----------------------------------------------------
        # 2.3. LEER LOS VUELOS CON PYSPARK
        # ----------------------------------------------------

        # Leemos el archivo completo con PySpark.
        # En Bronze mantenemos las columnas originales
        # como texto, sin limpieza ni muestreo.
        df_mes = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .csv(str(ruta_csv))
        )

        # ----------------------------------------------------
        # 2.4. AGREGAR METADATOS
        # ----------------------------------------------------

        # Estos campos permiten identificar el año,
        # el mes, el archivo y el momento de la ingesta.
        df_mes = (
            df_mes
            .withColumn("_source_year", F.lit(anio))
            .withColumn("_source_month", F.lit(mes))
            .withColumn(
                "_source_file",
                F.lit(ruta_csv.name)
            )
            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )
        )

        # ----------------------------------------------------
        # 2.5. VALIDAR EL ESQUEMA
        # ----------------------------------------------------

        # Comprobamos que los datos de 2022 y 2023
        # tengan las mismas columnas que nuestra tabla.
        columnas_nuevas = set(df_mes.columns)
        columnas_originales = set(columnas_bronze)

        if columnas_nuevas != columnas_originales:

            faltantes = columnas_originales - columnas_nuevas
            adicionales = columnas_nuevas - columnas_originales

            raise ValueError(
                f"Las columnas de {periodo} no coinciden. "
                f"Faltantes: {sorted(faltantes)}. "
                f"Adicionales: {sorted(adicionales)}."
            )

        # Organizamos las columnas en el orden original.
        df_mes = df_mes.select(*columnas_bronze)

        # ----------------------------------------------------
        # 2.6. GUARDAR EN DELTA LAKE
        # ----------------------------------------------------

        # APPEND agrega los vuelos nuevos y conserva
        # todos los registros de los meses anteriores.
        (
            df_mes.write
            .format("delta")
            .mode("append")
            .saveAsTable(TABLA_BRONZE)
        )

        # Solo registramos el periodo como cargado
        # después de completar correctamente la escritura.
        meses_cargados.add((anio, mes))

        duracion = round(time.time() - inicio, 2)

        print(f"Mes {periodo} cargado correctamente.")
        print(f"Tiempo de carga: {duracion} segundos")

        resultados.append({
            "periodo": periodo,
            "estado": "Cargado",
            "registros": None,
            "segundos": duracion
        })

    except Exception as error:

        print(f"ERROR en {periodo}: {error}")

        errores.append({
            "periodo": periodo,
            "error": str(error)
        })

        # Si aparece una diferencia en las columnas
        # o falla una descarga, detenemos el proceso
        # para revisar el problema antes de continuar.
        break


# ------------------------------------------------------------
# 3. RESUMEN DE INGESTA
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RESUMEN BRONZE BTS 2022-2025")
print("=" * 55)

# Contamos los vuelos por año y mes.
resumen_bronze = (
    spark.table(TABLA_BRONZE)
    .groupBy("_source_year", "_source_month")
    .agg(F.count("*").alias("total_vuelos"))
    .orderBy("_source_year", "_source_month")
)

display(resumen_bronze)

# Calculamos cuántos meses tenemos en total.
meses_finales = resumen_bronze.count()

print("Meses cargados:", meses_finales, "de 48")

# Sumamos los registros de todos los meses.
total_final = spark.table(TABLA_BRONZE).count()

print("Total de vuelos:", f"{total_final:,}")

# Mostramos el tiempo y estado de los meses procesados.
if resultados:
    display(resultados)

if errores:
    print("Errores pendientes:")
    display(errores)
else:
    print("La ejecución terminó sin errores detectados.")


In [0]:

# Validamos que estén presentes todos los meses de 2022 a 2025
# y comprobamos que ninguno tenga cero registros.

from pyspark.sql import functions as F

tabla = "workspace.bronze_flights.bts_flights_raw"

resumen = (
    spark.table(tabla)
    .groupBy("_source_year", "_source_month")
    .agg(F.count("*").alias("vuelos"))
)

esperados = {
    (anio, mes)
    for anio in range(2022, 2026)
    for mes in range(1, 13)
}

filas = resumen.collect()

encontrados = {
    (fila["_source_year"], fila["_source_month"])
    for fila in filas
}

faltantes = esperados - encontrados
meses_vacios = [
    (fila["_source_year"], fila["_source_month"])
    for fila in filas
    if fila["vuelos"] == 0
]

print("Periodos encontrados:", len(encontrados), "de 48")
print("Meses faltantes:", faltantes)
print("Meses sin vuelos:", meses_vacios)

if encontrados == esperados and not meses_vacios:
    print("VALIDACIÓN EXITOSA: cobertura mensual completa.")
else:
    print("REVISAR: se encontraron problemas de cobertura.")


In [0]:

# ============================================================
# CELDA 9. VALIDACIÓN DE DUPLICADOS POR ARCHIVO
# ============================================================

# Revisamos los archivos registrados en Bronze para comprobar
# que no existan nombres de archivo asociados a más de un mes.
# También verificamos que el periodo registrado en el nombre
# coincida con los metadatos de año y mes.

from pyspark.sql import functions as F

tabla = "workspace.bronze_flights.bts_flights_raw"

df = spark.table(tabla)

# Comparamos los archivos registrados con los periodos.
archivos = (
    df.groupBy(
        "_source_year",
        "_source_month",
        "_source_file"
    )
    .agg(F.count("*").alias("total_registros"))
    .orderBy("_source_year", "_source_month")
)

display(archivos)

# Revisamos que exista una sola combinación de archivo
# para cada año y mes.
archivos_por_mes = (
    archivos.groupBy("_source_year", "_source_month")
    .agg(F.count("*").alias("cantidad_archivos"))
)

problemas = archivos_por_mes.filter(
    F.col("cantidad_archivos") != 1
)

print("Periodos revisados:", archivos_por_mes.count())
print("Periodos con varios archivos:", problemas.count())

# Verificamos que el nombre del archivo sea el esperado.
nombre_esperado = F.format_string(
    "bts_%04d_%02d.csv",
    F.col("_source_year"),
    F.col("_source_month")
)

nombres_incorrectos = archivos.filter(
    F.col("_source_file") != nombre_esperado
)

print("Archivos con nombre incorrecto:", nombres_incorrectos.count())

if problemas.count() == 0 and nombres_incorrectos.count() == 0:
    print("VALIDACIÓN EXITOSA: metadatos de archivos consistentes.")
else:
    print("REVISAR: se encontraron inconsistencias.")


In [0]:

# ============================================================
# CELDA 10. REVISIÓN DE POSIBLES CARGAS DUPLICADAS
# ============================================================

# Comprobamos si un mismo mes contiene registros
# con diferentes marcas de tiempo de ingesta.
# Esto nos ayuda a identificar posibles cargas repetidas.

from pyspark.sql import functions as F

tabla = "workspace.bronze_flights.bts_flights_raw"

df = spark.table(tabla)

# Contamos cuántas marcas de ingesta distintas tiene
# cada mes del periodo 2022-2025.
revision = (
    df.groupBy("_source_year", "_source_month")
    .agg(
        F.count("*").alias("total_vuelos"),
        F.countDistinct("_ingestion_timestamp").alias(
            "marcas_ingesta"
        )
    )
    .orderBy("_source_year", "_source_month")
)

# Identificamos los periodos con más de una marca.
posibles_recargas = revision.filter(
    F.col("marcas_ingesta") > 1
)

print("Periodos analizados:", revision.count())
print(
    "Periodos con varias marcas de ingesta:",
    posibles_recargas.count()
)

display(posibles_recargas)

# Esta comprobación es una señal de posibles recargas,
# no una prueba definitiva de vuelos duplicados.
if posibles_recargas.count() == 0:
    print("Sin señales de recargas por marca de tiempo.")
else:
    print("Hay periodos que requieren revisión adicional.")
